# AML Alert Escalation Prediction

Предсказание вероятности того, что сигнал финансового мониторинга (alert) будет эскалирован (отправлен на полную проверку), по истории транзакций клиента.

Метрика — ROC-AUC. На выходе: вероятность (float в [0,1]), не бинарная метка.

**Итоговый результат кросс-валидации: ROC-AUC ≈ 0.626** (усреднение по 5 seed).

Честная оговорка: индивидуальная линейная корреляция каждого отдельного признака с целью не превышает 0.08 — сигнал слабый и распределён нелинейно по многим признакам сразу, а не сосредоточен в одном явном индикаторе. Это разумно ожидаемо для датасета без данных о контрагенте операции.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from pathlib import Path

DATA_DIR = Path("data") if Path("data").exists() else Path("../data")  # data/ в корне репо
SEEDS = [42, 1, 7, 123, 2026]

## 1. Загрузка исходных данных

In [ ]:
train_signals = pd.read_csv(DATA_DIR / "train_signals.csv")
test_signals = pd.read_csv(DATA_DIR / "test_signals.csv")
train_tx = pd.read_parquet(DATA_DIR / "train_transactions.parquet")
test_tx = pd.read_parquet(DATA_DIR / "test_transactions.parquet")

print(f"train_signals: {train_signals.shape}, test_signals: {test_signals.shape}")
print(f"train_tx: {train_tx.shape}, test_tx: {test_tx.shape}")

## 2. Feature engineering

Три группы признаков:
1. **Из ТЗ буквально** — количество операций, приход/расход, доли по типам, статистика по сумме, активность перед сигналом.
2. **Расширения** — чистый денежный поток, статистика по сумме на каждый тип отдельно, время суток / день недели, скорость операций.
3. **AML-паттерны** (добавлены после того, как v1/v2 дали AUC ~0.59-0.60, почти случайность) — интервалы между операциями (burst), "layering" (приход+расход похожего объёма в один день — классический паттерн быстрого перегона денег), всплеск активности перед сигналом, аномальность последней операции относительно собственной истории клиента.

In [ ]:
TXN_TYPES = ["bank_otkazmasi", "karta", "naqd", "xalqaro"]
RECENT_WINDOWS = [3, 7, 14]

def build_features(tx: pd.DataFrame, signals: pd.DataFrame) -> pd.DataFrame:
    tx = tx.merge(signals[["signal_id", "signal_sanasi"]], on="signal_id", how="left")
    tx["signal_sanasi"] = pd.to_datetime(tx["signal_sanasi"])
    tx["days_before_signal"] = (tx["signal_sanasi"] - tx["tranzaksiya_vaqti"]).dt.total_seconds() / 86400
    tx["hour"] = tx["tranzaksiya_vaqti"].dt.hour
    tx["weekday"] = tx["tranzaksiya_vaqti"].dt.weekday
    tx["is_night"] = tx["hour"].between(0, 5).astype(int)
    tx["is_weekend"] = (tx["weekday"] >= 5).astype(int)
    tx["date_only"] = tx["tranzaksiya_vaqti"].dt.date
    tx = tx.sort_values(["signal_id", "tranzaksiya_vaqti"])

    g = tx.groupby("signal_id")
    feats = pd.DataFrame(index=g.size().index)
    feats["n_txn"] = g.size()

    kirim_mask = tx["kirim_chiqim"] == "kirim"
    chiqim_mask = tx["kirim_chiqim"] == "chiqim"
    feats["n_kirim"] = tx[kirim_mask].groupby("signal_id").size().reindex(feats.index, fill_value=0)
    feats["n_chiqim"] = tx[chiqim_mask].groupby("signal_id").size().reindex(feats.index, fill_value=0)
    feats["kirim_chiqim_ratio"] = feats["n_kirim"] / (feats["n_chiqim"] + 1)
    feats["kirim_share"] = feats["n_kirim"] / feats["n_txn"]

    feats["kirim_sum"] = tx[kirim_mask].groupby("signal_id")["miqdor_indeksi"].sum().reindex(feats.index, fill_value=0)
    feats["chiqim_sum"] = tx[chiqim_mask].groupby("signal_id")["miqdor_indeksi"].sum().reindex(feats.index, fill_value=0)
    feats["net_flow"] = feats["kirim_sum"] - feats["chiqim_sum"]
    feats["turnover"] = feats["kirim_sum"].abs() + feats["chiqim_sum"].abs()
    feats["net_flow_ratio"] = feats["net_flow"] / (feats["turnover"] + 1e-6)

    for t in TXN_TYPES:
        sub = tx[tx["tranzaksiya_turi"] == t]
        cnt = sub.groupby("signal_id").size()
        feats[f"share_{t}"] = cnt.reindex(feats.index, fill_value=0) / feats["n_txn"]
        feats[f"mean_amt_{t}"] = sub.groupby("signal_id")["miqdor_indeksi"].mean().reindex(feats.index, fill_value=0)
        feats[f"sum_amt_{t}"] = sub.groupby("signal_id")["miqdor_indeksi"].sum().reindex(feats.index, fill_value=0)

    feats["amt_min"] = g["miqdor_indeksi"].min()
    feats["amt_max"] = g["miqdor_indeksi"].max()
    feats["amt_mean"] = g["miqdor_indeksi"].mean()
    feats["amt_std"] = g["miqdor_indeksi"].std().fillna(0)
    feats["amt_range"] = feats["amt_max"] - feats["amt_min"]
    feats["amt_median"] = g["miqdor_indeksi"].median()
    feats["amt_q25"] = g["miqdor_indeksi"].quantile(0.25)
    feats["amt_q75"] = g["miqdor_indeksi"].quantile(0.75)
    feats["amt_iqr"] = feats["amt_q75"] - feats["amt_q25"]
    feats["amt_cv"] = feats["amt_std"] / (feats["amt_mean"].abs() + 1e-6)

    feats["night_share"] = g["is_night"].mean()
    feats["weekend_share"] = g["is_weekend"].mean()
    feats["n_unique_hours"] = tx.groupby("signal_id")["hour"].nunique().reindex(feats.index, fill_value=0)

    for w in RECENT_WINDOWS:
        recent = tx[(tx["days_before_signal"] >= 0) & (tx["days_before_signal"] <= w)]
        rc = recent.groupby("signal_id").size()
        feats[f"n_txn_last_{w}d"] = rc.reindex(feats.index, fill_value=0)
        feats[f"share_txn_last_{w}d"] = feats[f"n_txn_last_{w}d"] / feats["n_txn"]

    feats["days_since_last_txn"] = g["days_before_signal"].min()
    feats["days_since_first_txn"] = g["days_before_signal"].max()
    feats["history_span_days"] = feats["days_since_first_txn"] - feats["days_since_last_txn"]

    feats["n_unique_days"] = tx.groupby("signal_id")["date_only"].nunique().reindex(feats.index, fill_value=1)
    feats["txn_velocity"] = feats["n_txn"] / feats["history_span_days"].clip(lower=1)
    feats["avg_gap_days"] = feats["history_span_days"] / feats["n_txn"].clip(lower=1)

    tx["prev_time"] = tx.groupby("signal_id")["tranzaksiya_vaqti"].shift(1)
    tx["gap_hours"] = (tx["tranzaksiya_vaqti"] - tx["prev_time"]).dt.total_seconds() / 3600
    gap_stats = tx.groupby("signal_id")["gap_hours"].agg(["min", "mean", "std"])
    feats["gap_min_hours"] = gap_stats["min"].reindex(feats.index)
    feats["gap_mean_hours"] = gap_stats["mean"].reindex(feats.index)
    feats["gap_std_hours"] = gap_stats["std"].reindex(feats.index).fillna(0)
    tx["is_burst"] = (tx["gap_hours"] < 1).astype(float)
    feats["burst_share"] = tx.groupby("signal_id")["is_burst"].mean().reindex(feats.index).fillna(0)

    per_day = tx.groupby(["signal_id", "date_only"]).size().groupby("signal_id").max()
    feats["max_txn_per_day"] = per_day.reindex(feats.index, fill_value=1)

    daily = tx.groupby(["signal_id", "date_only", "kirim_chiqim"])["miqdor_indeksi"].sum().unstack(fill_value=0)
    if "kirim" not in daily.columns: daily["kirim"] = 0
    if "chiqim" not in daily.columns: daily["chiqim"] = 0
    daily["both_present"] = (daily["kirim"] != 0) & (daily["chiqim"] != 0)
    daily["similarity"] = 1 - (daily["kirim"] - daily["chiqim"]).abs() / (daily["kirim"].abs() + daily["chiqim"].abs() + 1e-6)
    layering_days = daily[daily["both_present"] & (daily["similarity"] > 0.7)].groupby("signal_id").size()
    feats["layering_days"] = layering_days.reindex(feats.index, fill_value=0)
    feats["layering_share"] = feats["layering_days"] / feats["n_unique_days"]

    avg_daily_rate = feats["n_txn"] / feats["history_span_days"].clip(lower=1)
    recent3 = tx[(tx["days_before_signal"] >= 0) & (tx["days_before_signal"] <= 3)]
    recent3_cnt = recent3.groupby("signal_id").size().reindex(feats.index, fill_value=0)
    feats["spike_ratio_3d"] = (recent3_cnt / 3) / (avg_daily_rate + 1e-6)

    last_txn = tx.sort_values("tranzaksiya_vaqti").groupby("signal_id").tail(1).set_index("signal_id")
    feats["last_txn_amt"] = last_txn["miqdor_indeksi"].reindex(feats.index)
    feats["last_txn_zscore"] = (feats["last_txn_amt"] - feats["amt_mean"]) / (feats["amt_std"] + 1e-6)

    return feats.reset_index()

In [ ]:
train_feats = build_features(train_tx, train_signals)
train_feats = train_feats.merge(train_signals[["signal_id", "eskalatsiya"]], on="signal_id")
test_feats = build_features(test_tx, test_signals)
print(f"train_feats: {train_feats.shape}, test_feats: {test_feats.shape}")
train_feats.head()

## 3. EDA

In [ ]:
print(train_feats.describe())
print("\nбаланс классов:")
print(train_feats["eskalatsiya"].value_counts(normalize=True))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for cls, color, label in [(0, "steelblue", "не эскалирован"), (1, "crimson", "эскалирован")]:
    subset = train_feats[train_feats["eskalatsiya"] == cls]["amt_mean"]
    ax.hist(subset, bins=40, alpha=0.5, color=color, label=label, density=True)
ax.set_xlabel("amt_mean"); ax.set_ylabel("плотность")
ax.set_title("Распределение amt_mean по классам"); ax.legend()
plt.tight_layout(); plt.show()

## 4. Обучение модели

**Урок из промежуточных экспериментов:** агрессивный `scale_pos_weight` (полная компенсация дисбаланса ≈4.8) заставлял модель сходиться за 3-4 итерации и давал АУС ниже (0.593), чем вообще без него (0.621). Для ROC-AUC балансировка классов не обязательна так, как для accuracy/F1 — итоговая конфигурация её не использует, вместо этого — мягкая регуляризация (`reg_alpha`, `reg_lambda`, неглубокие деревья) и усреднение по 5 seed.

In [ ]:
feature_cols = [c for c in train_feats.columns if c not in ("signal_id", "eskalatsiya")]
X, y = train_feats[feature_cols], train_feats["eskalatsiya"]
X_test = test_feats[feature_cols]

BASE_PARAMS = dict(n_estimators=600, learning_rate=0.02, num_leaves=15, max_depth=4,
                    min_child_samples=30, subsample=0.7, colsample_bytree=0.6,
                    reg_alpha=1.0, reg_lambda=1.0, verbosity=-1)

all_oof = np.zeros((len(SEEDS), len(X)))
all_test_pred = np.zeros((len(SEEDS), len(X_test)))
seed_aucs = []

for si, seed in enumerate(SEEDS):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    oof = np.zeros(len(X))
    for tr_idx, val_idx in skf.split(X, y):
        model = lgb.LGBMClassifier(random_state=seed, **BASE_PARAMS)
        model.fit(X.iloc[tr_idx], y.iloc[tr_idx],
                  eval_set=[(X.iloc[val_idx], y.iloc[val_idx])], eval_metric="auc",
                  callbacks=[lgb.early_stopping(100, verbose=False)])
        oof[val_idx] = model.predict_proba(X.iloc[val_idx])[:, 1]
    auc = roc_auc_score(y, oof)
    seed_aucs.append(auc)
    all_oof[si] = oof
    print(f"seed {seed}: AUC = {auc:.4f}")

    final_model = lgb.LGBMClassifier(random_state=seed, **{**BASE_PARAMS, "n_estimators": 300})
    final_model.fit(X, y)
    all_test_pred[si] = final_model.predict_proba(X_test)[:, 1]

blended_oof = all_oof.mean(axis=0)
blended_auc = roc_auc_score(y, blended_oof)
print(f"\nсреднее по seed: {np.mean(seed_aucs):.4f} +- {np.std(seed_aucs):.4f}")
print(f"AUC усреднённого (blended) oof: {blended_auc:.4f}")

## 5. Формирование сдачи

In [ ]:
blended_test_pred = all_test_pred.mean(axis=0)
submission = pd.DataFrame({"signal_id": test_feats["signal_id"], "ehtimollik": blended_test_pred})

assert submission["signal_id"].duplicated().sum() == 0, "дубли signal_id"
assert submission["ehtimollik"].between(0, 1).all(), "вероятность вне [0,1]"
assert submission.isna().sum().sum() == 0, "есть пропуски"
assert len(submission) == 6000
assert set(submission["signal_id"]) == set(test_signals["signal_id"]), "signal_id в submission не совпадают с test_signals.csv"

submission.to_csv("team_E418F3DF.csv", index=False)
print(f"сохранено: team_E418F3DF.csv, строк: {len(submission)}")
submission.head()

## Итог

**ROC-AUC на кросс-валидации (усреднение 5 seed): ≈ 0.626.**

Путь к этому результату (честно, включая то, что не сработало):
1. Признаки буквально по ТЗ → AUC 0.594.
2. Расширенные агрегаты (net flow, время суток, статистика по типам) + `scale_pos_weight` → AUC 0.598, почти не изменилось — при этом отдельный тест показал, что `scale_pos_weight` сам по себе вредил (лучшая итерация модели наступала за 3-4 раунда вместо полноценного обучения).
3. Проверка линейной корреляции каждого признака с целью — максимум 0.08. Сигнал не сосредоточен в одном очевидном индикаторе.
4. Добавлены признаки из практики антифрода (burst, layering, всплеск активности, аномальность последней операции) + отказ от `scale_pos_weight` в пользу регуляризации → AUC 0.6205.
5. Усреднение по 5 seed для устойчивости оценки → **0.626**.

**Ограничение, которое честно нужно назвать:** индивидуальные признаки слабо коррелируют с целью, а данные не включают информацию о контрагенте операции — это структурно ограничивает достижимое качество по сравнению с продакшен-системами антифрода, где обычно есть граф связей между счетами.